In [2]:
import numpy as np
import math
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


data = load_breast_cancer()
X, y = data.data, data.target

n_features = X.shape[1]

def fitness(position):
    mask = 1 / (1 + np.exp(-position)) > 0.5

    if np.sum(mask) == 0:
        return 1.0

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    accuracy = cross_val_score(
        model, X[:, mask], y, cv=5,
        scoring='accuracy'
    ).mean()


    error = 1 - accuracy
    penalty = 0.01 * (np.sum(mask) / n_features)

    return error + penalty



def levy_flight(size):
    beta = 1.5
    sigma = (
        math.gamma(1 + beta) * np.sin(np.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta *
         2 ** ((beta - 1) / 2))
    ) ** (1 / beta)

    u = np.random.normal(0, sigma, size)
    v = np.random.normal(0, 1, size)

    return u / (np.abs(v) ** (1 / beta))


n_nests = 10
iterations = 20
pa = 0.25
alpha = 0.01


nests = np.random.uniform(-1, 1,
                          (n_nests, n_features))

scores = np.array([fitness(n) for n in nests])

best_idx = np.argmin(scores)
best_nest = nests[best_idx].copy()
best_score = scores[best_idx]


for t in range(iterations):

    for i in range(n_nests):


        new_nest = nests[i] + alpha * levy_flight(n_features)

        new_nest = np.clip(new_nest, -6, 6)
        new_score = fitness(new_nest)


        j = np.random.randint(n_nests)

        if new_score < scores[j]:
            nests[j] = new_nest
            scores[j] = new_score


    n_abandon = int(pa * n_nests)

    worst = np.argsort(scores)[-n_abandon:]

    for j in worst:
        nests[j] = np.random.uniform(
            -1, 1, n_features
        )
        scores[j] = fitness(nests[j])


    best_idx = np.argmin(scores)

    if scores[best_idx] < best_score:
        best_score = scores[best_idx]
        best_nest = nests[best_idx].copy()

    print(f"Iteration {t+1}: Fitness = {best_score:.4f}")

selected = 1 / (1 + np.exp(-best_nest)) > 0.5

print("\nSelected feature indices:")
print(np.where(selected)[0])

print("\nSelected feature names:")
print(data.feature_names[selected])

print("\nNumber of selected features:",
      np.sum(selected))

print("Best fitness:", best_score)


Iteration 1: Fitness = 0.0296
Iteration 2: Fitness = 0.0296
Iteration 3: Fitness = 0.0264
Iteration 4: Fitness = 0.0243
Iteration 5: Fitness = 0.0243
Iteration 6: Fitness = 0.0243
Iteration 7: Fitness = 0.0243
Iteration 8: Fitness = 0.0243
Iteration 9: Fitness = 0.0243
Iteration 10: Fitness = 0.0243
Iteration 11: Fitness = 0.0243
Iteration 12: Fitness = 0.0243
Iteration 13: Fitness = 0.0243
Iteration 14: Fitness = 0.0243
Iteration 15: Fitness = 0.0243
Iteration 16: Fitness = 0.0243
Iteration 17: Fitness = 0.0243
Iteration 18: Fitness = 0.0243
Iteration 19: Fitness = 0.0243
Iteration 20: Fitness = 0.0243

Selected feature indices:
[ 1  3  4  7 11 12 16 18 23 24 25 26 27 28 29]

Selected feature names:
['mean texture' 'mean area' 'mean smoothness' 'mean concave points'
 'texture error' 'perimeter error' 'concavity error' 'symmetry error'
 'worst area' 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']

Number of se